# The lab

A notebook running *inside* a container that has the volume mounted at
`/storage`. Nothing below is different from the same code on your laptop
except that `lab.ROOT` is the volume itself rather than a copy of it pulled
down by hand -- so `Artifact.at` finds real files, `bind` loads real state,
and declaring is a local call instead of `declare_fn.remote(...)`.

Your edits here live on the volume (`/storage/notebooks`) and are committed
every 30s, so they survive this container. This file is seeded from the repo
and never overwritten -- copy it per experiment rather than editing it in
place if you want the original back.

## What's on the volume

In [ ]:
import lab

lab.refresh()  # pull in whatever other containers have committed since boot
lab.ls("tokenizers")

## Load an artifact by pointing at its path

`lab.load` is `Artifact.at`: the manifest in the folder says which artifact
this is, the folder itself says where its root is. It comes back **bound**
when its files are all there -- for a tokenizer that means vocab and merges
loaded, ready to `encode`.

In [ ]:
# Paste any path from the listing above. ls() also lists the tokenized
# sources nested under each tokenizer, so [0] is the tokenizer itself.
tokenizer = lab.load(lab.ls("tokenizers")[0])
tokenizer

In [ ]:
tokenizer.encode("the sea was angry that day")[:20]

## ...or by writing its parameters out

The same object, the other spelling. An artifact's path under a root is a
pure function of its parameters, so naming the parameters and naming the
path land in exactly the same place -- pick whichever you have to hand.

In [ ]:
from sources.artifact import Source
from tokenizers.bpe import Tokenizer as BPETokenizer

odyssey = Source(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")

same = BPETokenizer(
    vocab_size=3000,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick),
)
same.artifact_path  # compare against what ls() printed

In [ ]:
same.bind(lab.ROOT)  # raises FileNotFoundError if it isn't built yet

## Declare a run

Build the graph the way [declare.ipynb](../declare.ipynb) does, then check it
against the volume. `lab.check` writes nothing: everything shared should read
`done` (or `declared`), everything new to this run should read `new`.

In [ ]:
from dag.artifact import Resources
from datasets.artifact import DataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig

RUN_ID = "RUN_LAB"  # <- change this per run

dataset = DataSet.from_sources(
    run_id=RUN_ID,
    tokenizer=same,
    train_sources=[mobydick],
    valid_sources=[odyssey],
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=dataset,
    tokenizer=same,
    model_parameters=ModelParameters(hidden_size=64, num_layers=2),
    config=PretrainingConfig(
        total_steps=2000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
    ),
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

lab.check(pretraining)  # preview, no writes

In [ ]:
lab.plan(pretraining)  # the same tree, drawn -- outline color is status

In [ ]:
lab.declare(pretraining)  # writes every `new` manifest, then commits

Declared, not produced: a `declared` row has a manifest and no files yet.
The run and its artifacts now show up on the dashboard, and each one gets a
launch button once nothing is blocking it.

Anything else you write under `lab.ROOT` from a cell stays on this container
until you call `lab.publish()` -- `lab.declare` already did.